# SkillOpt: Self-Evolving Agent Skills Tutorial

**Based on:** arXiv:2605.23904v2 "SkillOpt: Executive Strategy for Self-Evolving Agent Skills"

This notebook demonstrates how to use SkillOpt to optimize agent skills through text-space optimization.

## Table of Contents
1. [Setup](#setup)
2. [Core Concepts](#concepts)
3. [Basic Example](#basic)
4. [Advanced: Real LLM Integration](#advanced)
5. [Applications](#applications)
6. [Results Analysis](#results)

<a id='setup'></a>
## 1. Setup

First, let's import the necessary libraries and our SkillOpt implementation.

In [ ]:
import sys
import os

# Add current directory to path
sys.path.insert(0, os.getcwd())

from skillopt_implementation import (
    SkillOptimizer,
    SkillDocument,
    Edit,
    EditType,
    Trajectory,
)

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports successful")

<a id='concepts'></a>
## 2. Core Concepts

### The Deep Learning Analogy

SkillOpt treats a skill document as the **external trainable state** of a frozen agent:

| Deep Learning | SkillOpt |
|--------------|----------|
| Parameters | Skill document |
| Gradient direction | Trajectory-derived edits |
| Learning rate | Edit budget (max edits/step) |
| Validation check | Held-out selection gate |
| Batch/minibatch | Rollout batches, reflection minibatches |

### Key Components

1. **Rollout Batch**: Execute tasks with current skill
2. **Minibatch Reflection**: Analyze failures/successes to find patterns
3. **Edit Operations**: APPEND, INSERT_AFTER, REPLACE, DELETE
4. **Edit Budget**: Learning rate (starts high, decays)
5. **Validation Gate**: Accept only if score improves
6. **Rejected-Edit Buffer**: Negative feedback from failed edits
7. **Slow/Meta Update**: Epoch-boundary longitudinal insights

<a id='basic'></a>
## 3. Basic Example: Math Problem Solving

Let's start with a simple example: optimizing a skill for solving math word problems.

In [ ]:
# Initial skill document
initial_math_skill = SkillDocument(
    content="""# Math Problem Solving

## Core Approach
Read the problem and solve it step by step.

## Key Guidelines
- Identify what is being asked
- Extract relevant numbers
- Perform calculations
"""
)

print("Initial Skill:")
print("="*60)
print(initial_math_skill.content)
print("="*60)

In [ ]:
# Mock target model for math problems
def math_target_model(skill: str, task: dict) -> dict:
    """
    Simulates an LLM solving math problems with the given skill.
    In practice, this would call GPT-4, Claude, etc.
    """
    problem = task.get("input", "")
    correct_answer = task.get("answer", "")
    
    # Simulate agent behavior based on skill content
    skill_lower = skill.lower()
    
    # Check if skill mentions important strategies
    has_equation_strategy = "equation" in skill_lower or "formula" in skill_lower
    has_verification = "verify" in skill_lower or "check" in skill_lower
    has_unit_handling = "unit" in skill_lower or "label" in skill_lower
    
    # Simplified scoring logic
    base_score = 0.4
    if has_equation_strategy:
        base_score += 0.3
    if has_verification:
        base_score += 0.2
    if has_unit_handling and "units" in problem.lower():
        base_score += 0.1
    
    # Add randomness
    score = min(base_score + np.random.uniform(-0.1, 0.1), 1.0)
    success = score > 0.75
    
    output = f"Answer: {correct_answer if success else 'incorrect'}" 
    
    return {
        "output": output,
        "success": success,
        "score": score,
        "steps": [],
        "metadata": {},
    }

print("✓ Math target model defined")

In [ ]:
# Mock optimizer model
def math_optimizer_model(prompt: str) -> str:
    """
    Simulates an optimizer LLM that analyzes trajectories.
    In practice, this would call a stronger model like GPT-4 or Claude Opus.
    """
    if "failed" in prompt.lower() or "failures" in prompt.lower():
        # Propose corrective edits for failures
        return """EDIT_TYPE: APPEND
CONTENT: ## Advanced Strategies\n\n- Write equations explicitly before solving\n- Always verify the answer makes sense in context
---
EDIT_TYPE: INSERT_AFTER
CONTENT: - Pay attention to units and convert if necessary
ANCHOR: - Perform calculations
---"""
    else:
        # Propose reinforcement edits for successes
        return """EDIT_TYPE: APPEND
CONTENT: ## What Works\n\n- The systematic step-by-step approach is effective
---"""

print("✓ Math optimizer model defined")

In [ ]:
# Create training data
train_math_tasks = [
    {"id": f"train_{i}", "input": f"If Alice has {i} apples and gives away {i//2}, how many does she have left?", "answer": str(i - i//2)}
    for i in range(2, 22, 2)
] + [
    {"id": f"train_multi_{i}", "input": f"A rectangle is {i} meters long and {i+2} meters wide. What is the area in square meters?", "answer": str(i * (i+2))}
    for i in range(3, 13)
]

validation_math_tasks = [
    {"id": f"val_{i}", "input": f"Bob has {i*3} dollars and spends {i}, how much is left?", "answer": str(i*3 - i)}
    for i in range(5, 15)
]

print(f"✓ Created {len(train_math_tasks)} training tasks and {len(validation_math_tasks)} validation tasks")

In [ ]:
# Create and run optimizer
math_optimizer = SkillOptimizer(
    target_model=math_target_model,
    optimizer_model=math_optimizer_model,
    rollout_batch_size=10,
    reflection_minibatch_size=5,
    edit_budget_initial=3,
    edit_budget_floor=1,
    edit_budget_schedule="cosine",
    epochs=3,
    slow_update_samples=10,
    use_rejected_buffer=True,
)

print("Running optimization...\n")
best_math_skill, math_history = math_optimizer.optimize(
    initial_skill=initial_math_skill,
    train_tasks=train_math_tasks,
    validation_tasks=validation_math_tasks,
)

In [ ]:
# Display optimized skill
print("\n" + "="*80)
print("OPTIMIZED MATH SKILL:")
print("="*80)
print(best_math_skill.content)
print("="*80)
print(f"\nFinal Validation Score: {best_math_skill.validation_score:.4f}")
print(f"Skill Version: {best_math_skill.version}")

<a id='advanced'></a>
## 4. Advanced: Real LLM Integration

Here's how to integrate with real LLM APIs (OpenAI, Anthropic, etc.)

In [ ]:
# Uncomment and configure with your API keys

# import openai
# import anthropic

# # Option 1: OpenAI (GPT-4)
# openai_client = openai.OpenAI(api_key="your-api-key")

# def gpt4_target_model(skill: str, task: dict) -> dict:
#     """Real GPT-4 target model."""
#     response = openai_client.chat.completions.create(
#         model="gpt-4",
#         messages=[
#             {"role": "system", "content": skill},
#             {"role": "user", "content": task["input"]}
#         ],
#         max_tokens=500,
#     )
#     
#     output = response.choices[0].message.content
#     
#     # Scoring logic (task-specific)
#     correct_answer = task.get("answer", "")
#     success = correct_answer.lower() in output.lower()
#     score = 1.0 if success else 0.0
#     
#     return {
#         "output": output,
#         "success": success,
#         "score": score,
#         "steps": [],
#         "metadata": {"model": "gpt-4"},
#     }

# def gpt4_optimizer_model(prompt: str) -> str:
#     """Real GPT-4 optimizer model."""
#     response = openai_client.chat.completions.create(
#         model="gpt-4",
#         messages=[
#             {"role": "system", "content": "You are a skill optimization expert. Analyze trajectories and propose precise edits."},
#             {"role": "user", "content": prompt}
#         ],
#         max_tokens=1000,
#     )
#     return response.choices[0].message.content


# # Option 2: Anthropic (Claude)
# anthropic_client = anthropic.Anthropic(api_key="your-api-key")

# def claude_target_model(skill: str, task: dict) -> dict:
#     """Real Claude target model."""
#     response = anthropic_client.messages.create(
#         model="claude-opus-4-5",
#         max_tokens=500,
#         system=skill,
#         messages=[
#             {"role": "user", "content": task["input"]}
#         ],
#     )
#     
#     output = response.content[0].text
#     
#     # Scoring logic (task-specific)
#     correct_answer = task.get("answer", "")
#     success = correct_answer.lower() in output.lower()
#     score = 1.0 if success else 0.0
#     
#     return {
#         "output": output,
#         "success": success,
#         "score": score,
#         "steps": [],
#         "metadata": {"model": "claude-opus-4-5"},
#     }

# def claude_optimizer_model(prompt: str) -> str:
#     """Real Claude optimizer model."""
#     response = anthropic_client.messages.create(
#         model="claude-opus-4-5",
#         max_tokens=1500,
#         system="You are a skill optimization expert. Analyze trajectories and propose precise, bounded edits.",
#         messages=[
#             {"role": "user", "content": prompt}
#         ],
#     )
#     return response.content[0].text

print("✓ Real LLM integration templates defined (commented out)")
print("  To use: Uncomment, add your API keys, and replace mock models")

<a id='applications'></a>
## 5. Applications

### Documented Use Cases from the Paper

SkillOpt achieved significant improvements across multiple domains:

| Domain | Task | Baseline | SkillOpt | Gain |
|--------|------|----------|----------|------|
| Search | SearchQA | 77.7 | 87.3 | +9.6 |
| Spreadsheets | SpreadsheetBench | 41.8 | 80.7 | +38.9 |
| Documents | OfficeQA | 33.1 | 72.1 | +39.0 |
| Vision | DocVQA | 78.8 | 91.2 | +12.4 |
| Math | LiveMathBench | 37.6 | 66.9 | +29.3 |
| Embodied | ALFWorld | 83.6 | 95.5 | +11.9 |

### Example Learned Rules

**SearchQA:**  
*"Infer the expected answer type from clue wording, then choose the shortest canonical entity supported by co-occurring distinctive evidence."*

**SpreadsheetBench:**  
*"Inspect workbook structure and formulas, then write evaluated static values across the full requested target range instead of relying on Excel recalculation."*

**LiveMathBench:**  
*"In strongest-statement MCQs, rank choices by theorem strength and prefer a justified stronger-result option over true but weaker corollaries."*

<a id='results'></a>
## 6. Results Analysis

Let's visualize the optimization process.

In [ ]:
# Convert history to DataFrame
history_df = pd.DataFrame(math_history)

# Plot optimization trajectory
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Validation score over steps
axes[0].plot(history_df.index, history_df['validation_score'], marker='o', linewidth=2)
axes[0].axhline(y=initial_math_skill.validation_score, color='r', linestyle='--', label='Initial')
axes[0].set_xlabel('Optimization Step', fontsize=12)
axes[0].set_ylabel('Validation Score', fontsize=12)
axes[0].set_title('SkillOpt Optimization Trajectory', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Acceptance rate per epoch
acceptance_by_epoch = history_df.groupby('epoch')['accepted'].mean()
axes[1].bar(acceptance_by_epoch.index, acceptance_by_epoch.values, color='steelblue', alpha=0.7)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Acceptance Rate', fontsize=12)
axes[1].set_title('Edit Acceptance Rate by Epoch', fontsize=14, fontweight='bold')
axes[1].set_ylim([0, 1])
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nOptimization Statistics:")
print(f"  Total steps: {len(history_df)}")
print(f"  Edits accepted: {history_df['accepted'].sum()}")
print(f"  Acceptance rate: {history_df['accepted'].mean():.2%}")
print(f"  Final score: {history_df['validation_score'].iloc[-1]:.4f}")
print(f"  Improvement: {history_df['validation_score'].iloc[-1] - history_df['validation_score'].iloc[0]:.4f}")

## Key Takeaways

1. **Text-space optimization works**: Skills are optimized like neural network weights
2. **Validation gating is critical**: Prevents harmful edits from accumulating
3. **Edit economy**: Only 1-4 accepted edits can produce massive gains
4. **Transferable**: Skills transfer across models and execution harnesses
5. **Compact**: Final skills are 300-2000 tokens (median ~920)

## Next Steps

1. **Try on your domain**: Adapt the templates to your specific use case
2. **Integrate real LLMs**: Replace mock models with GPT-4, Claude, etc.
3. **Tune hyperparameters**: Adjust batch sizes, edit budgets for your data
4. **Build skill libraries**: Create reusable skills for common tasks
5. **Monitor drift**: Track skill performance over time and re-optimize

## Resources

- **Paper**: arXiv:2605.23904v2 "SkillOpt: Executive Strategy for Self-Evolving Agent Skills"
- **One-Pager**: `SkillOpt_Executive_OnePager.md` in this repository
- **Implementation**: `skillopt_implementation.py` in this repository